<a href="https://colab.research.google.com/github/jjmin2/CV_Project_9team/blob/main/CV_Evaluation_Comparison_of_AUROC_and_IoU.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import sys

sys.path.append('/content/drive/MyDrive/CV/activeLearning')
sys.path.append('/content/drive/MyDrive/CV/activeLearning/sobel_detector.py')
sys.path.append('/content/drive/MyDrive/CV/activeLearning/padim.py')

In [4]:
import os
import glob
import torch
import numpy as np
from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms

# 1. 데이터셋 클래스 선언
class MVTecDataset(Dataset):
    def __init__(self, data_dir, category, split='train'):
        self.data_dir = os.path.join(data_dir, category, split)
        self.split = split
        self.image_paths = sorted(glob.glob(os.path.join(self.data_dir, '**', '*.png'), recursive=True))

        self.transform = transforms.Compose([
            transforms.Resize((256, 256)),
            transforms.ToTensor()
        ])
        self.mask_transform = transforms.Compose([
            transforms.Resize((256, 256), interpolation=transforms.InterpolationMode.NEAREST),
            transforms.ToTensor()
        ])

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB')
        img_tensor = self.transform(image)

        if self.split == 'train':
            label = 0
            mask_tensor = torch.zeros((1, 256, 256))
        else:
            defect_type = os.path.basename(os.path.dirname(img_path))
            if defect_type == 'good':
                label = 0
                mask_tensor = torch.zeros((1, 256, 256))
            else:
                label = 1
                mask_path = img_path.replace('test', 'ground_truth').replace('.png', '_mask.png')
                if os.path.exists(mask_path):
                    mask_tensor = self.mask_transform(Image.open(mask_path).convert('L'))
                else:
                    mask_tensor = torch.zeros((1, 256, 256))

        return img_tensor, torch.tensor(label, dtype=torch.float32), mask_tensor, img_path

DATA_DIR = '/content/drive/MyDrive/CV/activeLearning/mvtec_anomaly_detection'
CATEGORY = 'bottle'

dataset_train = MVTecDataset(DATA_DIR, CATEGORY, split='train')
dataset_val = MVTecDataset(DATA_DIR, CATEGORY, split='val')
dataset_test = MVTecDataset(DATA_DIR, CATEGORY, split='test')

In [6]:
import os
import glob
import json
import time
import numpy as np
import pandas as pd
import cv2
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.metrics import roc_auc_score

# 팀원 모듈 임포트
from sobel_detector import SobelDetector
from padim import PaDiMDetector

# ==========================================
# 0. 평가지표 및 헬퍼 함수
# ==========================================
def calculate_auroc(y_true, y_scores): return roc_auc_score(y_true, y_scores)
def calculate_pixel_auroc(mask_true, score_map): return roc_auc_score(mask_true.flatten(), score_map.flatten())
def calculate_iou(mask_true, mask_pred):
    intersection, union = np.logical_and(mask_true, mask_pred).sum(), np.logical_or(mask_true, mask_pred).sum()
    return np.nan if union == 0 else intersection / union
def tensor_to_bgr(tensor):
    img_rgb = (tensor.squeeze(0).permute(1, 2, 0).numpy() * 255).astype(np.uint8)
    return cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)

# ==========================================
# 1. 데이터셋 클래스
# ==========================================
class MVTecDataset(Dataset):
    def __init__(self, data_dir, category, split='train', split_json=None):
        self.split = split
        self.transform = transforms.Compose([transforms.Resize((256, 256)), transforms.ToTensor()])
        self.mask_transform = transforms.Compose([transforms.Resize((256, 256), interpolation=transforms.InterpolationMode.NEAREST), transforms.ToTensor()])

        self.image_paths = []

        if split in ['train', 'val']:
            if split_json and os.path.exists(split_json):
                with open(split_json, 'r') as f:
                    splits = json.load(f)

                # 웹앱 JSON 기준 키 매핑
                json_key = 'statistics_images' if split == 'train' else 'validation_images'
                rel_paths = splits.get(json_key, [])

                # JSON에 명시된 경로를 절대 경로로 바로 연결
                for rel_p in rel_paths:
                    full_path = os.path.join(data_dir, category, rel_p)
                    self.image_paths.append(full_path)
            else:
                # JSON 파일이 없으면 기존처럼 전체 폴더 로드
                base_path = os.path.join(data_dir, category, 'train')
                self.image_paths = sorted(glob.glob(os.path.join(base_path, '**', '*.png'), recursive=True))

        elif split == 'test':
            base_path = os.path.join(data_dir, category, 'test')
            self.image_paths = sorted(glob.glob(os.path.join(base_path, '**', '*.png'), recursive=True))

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        img_tensor = self.transform(Image.open(img_path).convert('RGB'))

        if self.split in ['train', 'val']:
            label, mask_tensor = 0, torch.zeros((1, 256, 256))
        else:
            defect_type = os.path.basename(os.path.dirname(img_path))
            if defect_type == 'good':
                label, mask_tensor = 0, torch.zeros((1, 256, 256))
            else:
                label = 1
                mask_path = img_path.replace('test', 'ground_truth').replace('.png', '_mask.png')
                if os.path.exists(mask_path):
                    mask_tensor = self.mask_transform(Image.open(mask_path).convert('L'))
                else:
                    mask_tensor = torch.zeros((1, 256, 256))
        return img_tensor, torch.tensor(label, dtype=torch.float32), mask_tensor, img_path

# ==========================================
# 2. 경로 설정 및 모델 세팅
# ==========================================
print("1. 데이터셋 및 모델 세팅 시작...")
DATA_DIR = '/content/drive/MyDrive/CV/activeLearning/mvtec_anomaly_detection'
JSON_PATH = '/content/drive/MyDrive/CV/activeLearning/bottle.json'
CATEGORY = 'bottle'

dataset_train = MVTecDataset(DATA_DIR, CATEGORY, split='train', split_json=JSON_PATH)
dataset_val = MVTecDataset(DATA_DIR, CATEGORY, split='val', split_json=JSON_PATH)
dataset_test = MVTecDataset(DATA_DIR, CATEGORY, split='test')

print(f"- 학습(train): {len(dataset_train)}장, 검증(val): {len(dataset_val)}장 로드 완료 (웹앱과 동일)")

# --- Sobel 세팅 ---
sobel_model = SobelDetector()
train_images = [dataset_train[i][0] for i in range(len(dataset_train))]
val_images = [dataset_val[i][0] for i in range(len(dataset_val))]
sobel_model.fit(train_images)
sobel_model.calibrate(val_images)
print("- [Sobel] 정상 통계 추정 및 임계값 보정 완료")

# --- PaDiM 세팅 ---
padim_model = PaDiMDetector()
train_pt_path = '/content/drive/MyDrive/bottle_train_features.pt'
padim_model.fit_from_file(train_pt_path)

val_score_maps_padim = [padim_model.score_map(tensor_to_bgr(img)) for img in val_images]
padim_pixel_threshold = np.percentile(np.stack(val_score_maps_padim), 99.5)
print(f"- [PaDiM] 정상 통계 로드 및 픽셀 임계값({padim_pixel_threshold:.4f}) 보정 완료\n")

# ==========================================
# 3. 테스트 데이터 통합 평가 진행
# ==========================================
print("2. 테스트 데이터 통합 평가 진행 중...")
results = {
    'Sobel': {'y_true': [], 'y_scores': [], 'px_auroc': [], 'iou': [], 'time': []},
    'PaDiM': {'y_true': [], 'y_scores': [], 'px_auroc': [], 'iou': [], 'time': []}
}

for img_tensor, label, mask_tensor, _ in DataLoader(dataset_test, batch_size=1, shuffle=False):
    single_img, img_bgr, mask_numpy, lbl = img_tensor.squeeze(0), tensor_to_bgr(img_tensor), mask_tensor.squeeze().numpy(), label.item()
    results['Sobel']['y_true'].append(lbl)
    results['PaDiM']['y_true'].append(lbl)

    # Sobel
    sobel_res = sobel_model.predict(single_img)
    results['Sobel']['time'].append(sobel_res["elapsed_ms"])
    results['Sobel']['y_scores'].append(sobel_res["image_score"])

    # PaDiM
    start_time = time.perf_counter()
    padim_score_map = padim_model.score_map(img_bgr)
    results['PaDiM']['time'].append((time.perf_counter() - start_time) * 1000)
    results['PaDiM']['y_scores'].append(float(np.max(padim_score_map)))

    # 픽셀 평가 (불량일 경우 마스크 비교)
    if len(np.unique(mask_numpy)) > 1:
        results['Sobel']['px_auroc'].append(calculate_pixel_auroc(mask_numpy, sobel_res["score_map"]))
        results['PaDiM']['px_auroc'].append(calculate_pixel_auroc(mask_numpy, padim_score_map))
    if lbl == 1:
        sobel_iou = calculate_iou(mask_numpy, sobel_res["mask"])
        if not np.isnan(sobel_iou): results['Sobel']['iou'].append(sobel_iou)

        padim_pred_mask = (padim_score_map > padim_pixel_threshold).astype(np.uint8)
        padim_iou = calculate_iou(mask_numpy, padim_pred_mask)
        if not np.isnan(padim_iou): results['PaDiM']['iou'].append(padim_iou)

# ==========================================
# 4. 최종 지표 산출 및 통합 CSV 저장
# ==========================================
final_data = []
print("-" * 50)
for method in ['Sobel', 'PaDiM']:
    r = results[method]
    img_auroc, px_auroc = calculate_auroc(r['y_true'], r['y_scores']), np.mean(r['px_auroc']) if r['px_auroc'] else 0
    mean_iou, mean_time = np.mean(r['iou']) if r['iou'] else 0, np.mean(r['time'])
    print(f"[{method}] Image AUROC: {img_auroc:.4f} | Pixel AUROC: {px_auroc:.4f} | Mean IoU: {mean_iou:.4f} | Time: {mean_time:.2f} ms")
    final_data.append({'Method': method, 'Image AUROC': img_auroc, 'Pixel AUROC (Mean)': px_auroc, 'Mean IoU (Defective only)': mean_iou, 'Processing Time Mean (ms)': mean_time})

df_combined = pd.DataFrame(final_data)
df_combined.to_csv('/content/drive/MyDrive/Evaluation_Results_WebAligned.csv', index=False)
print("\n 결과가 구글 드라이브(Evaluation_Results_WebAligned.csv)에 저장되었습니다")

1. 데이터셋 및 모델 세팅 시작...
- 학습(train): 150장, 검증(val): 38장 로드 완료 (웹앱과 동일)
- [Sobel] 정상 통계 추정 및 임계값 보정 완료
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 143MB/s]


- [PaDiM] 정상 통계 로드 및 픽셀 임계값(8.2286) 보정 완료

2. 테스트 데이터 통합 평가 진행 중...
--------------------------------------------------
[Sobel] Image AUROC: 0.9817 | Pixel AUROC: 0.7951 | Mean IoU: 0.2019 | Time: 4.36 ms
[PaDiM] Image AUROC: 0.9873 | Pixel AUROC: 0.9818 | Mean IoU: 0.3380 | Time: 138.59 ms

 결과가 구글 드라이브(Evaluation_Results_WebAligned.csv)에 저장되었습니다


In [11]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore') # 시각화 시 발생하는 불필요한 경고 숨김

# ==========================================
# 0. 저장 폴더 세팅
# ==========================================
base_save_dir = '/content/drive/MyDrive/CV/activeLearning/PPT_Visualizations'
case_dir = os.path.join(base_save_dir, 'Cases')       # 일반 사례 비교
fail_dir = os.path.join(base_save_dir, 'Failures')    # 실패/엇갈림 사례
os.makedirs(case_dir, exist_ok=True)
os.makedirs(fail_dir, exist_ok=True)

# ==========================================
# 1. 정량 비교: AUROC, IoU, CPU 시간 그래프
# ==========================================
print("1. 정량 비교 그래프 생성 중...")
csv_path = '/content/drive/MyDrive/Evaluation_Results_WebAligned.csv'

if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    plt.style.use('seaborn-v0_8-whitegrid')
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle('Quantitative Comparison: Sobel vs PaDiM', fontsize=16, fontweight='bold')

    metrics = [
        ('Image AUROC', 'Image-level Anomaly Detection (AUROC)', axes[0, 0]),
        ('Pixel AUROC (Mean)', 'Pixel-level Segmentation (AUROC)', axes[0, 1]),
        ('Mean IoU (Defective only)', 'Defect Localization Accuracy (Mean IoU)', axes[1, 0]),
        ('Processing Time Mean (ms)', 'Average Processing Time (ms)', axes[1, 1])
    ]

    colors = ['#4C72B0', '#DD8452'] # 파란색(Sobel), 주황색(PaDiM)
    for col_name, title, ax in metrics:
        sns.barplot(data=df, x='Method', y=col_name, ax=ax, palette=colors)
        ax.set_title(title, fontsize=13)
        ax.set_ylabel('')
        ax.set_xlabel('')
        for p in ax.patches:
            height = p.get_height()
            text = f'{height:.2f} ms' if 'Time' in col_name else f'{height:.4f}'
            if 'Time' not in col_name: ax.set_ylim(0, 1.15)
            ax.text(p.get_x() + p.get_width() / 2., height + (height * 0.02), text, ha='center', va='bottom', fontweight='bold')

    plt.tight_layout()
    plt.savefig(os.path.join(base_save_dir, '1_Quantitative_Graphs.png'), dpi=300)
    plt.close()
    print(" -> 정량 비교 그래프(1_Quantitative_Graphs.png) 저장 완료.\n")
else:
    print(" -> [주의] CSV 파일이 없어 정량 비교 그래프 생성을 건너뜁니다.\n")

# ==========================================
# 2. 정성 비교: 일반 사례 및 실패 사례 이미지 추출
# ==========================================
print("2. 사례 비교 및 실패 사례 시각화 생성 중...")

# 검증(val) 데이터로 '이미지 단위' 임계값(Threshold) 도출
print(" - 각 모델별 최적 이미지 임계값 계산 중...")
sobel_val_scores = [sobel_model.predict(dataset_val[i][0].squeeze(0))["image_score"] for i in range(len(dataset_val))]
padim_val_scores = [np.max(padim_model.score_map(tensor_to_bgr(dataset_val[i][0]))) for i in range(len(dataset_val))]

# 정상(val) 이미지들이 받은 점수 중 가장 높은 점수를 임계값으로 설정
sobel_image_thresh = np.max(sobel_val_scores)
padim_image_thresh = np.max(padim_val_scores)
print(f" - 적용된 이미지 임계값 -> Sobel: {sobel_image_thresh:.4f}, PaDiM: {padim_image_thresh:.4f}")

# 추출할 이미지 개수 설정
max_type_samples = 2
max_fail_samples = 3

collected_types = {}
collected_fails = {'FP': 0, 'FN': 0, 'DIFF': 0}

for i in range(len(dataset_test)):
    types_done = sum(collected_types.values()) >= 8
    fails_done = all(v >= max_fail_samples for v in collected_fails.values())
    if types_done and fails_done:
        break

    img_tensor, label, mask_tensor, img_path = dataset_test[i]
    defect_type = os.path.basename(os.path.dirname(img_path))
    gt_label = int(label.item())

    single_img, img_bgr = img_tensor.squeeze(0), tensor_to_bgr(img_tensor)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    mask_numpy = mask_tensor.squeeze().numpy()

    # 모델 예측
    sobel_res = sobel_model.predict(single_img)
    padim_score_map = padim_model.score_map(img_bgr)

    # [핵심 수정] 예측 라벨 산출 (픽셀이 아닌 전체 이미지 점수 기준)
    sobel_pred = 1 if sobel_res["image_score"] > sobel_image_thresh else 0
    padim_pred = 1 if np.max(padim_score_map) > padim_image_thresh else 0

    save_targets = []

    # --- (A) 정상 및 결함 유형별 일반 사례 수집 ---
    if not types_done and collected_types.get(defect_type, 0) < max_type_samples:
        collected_types[defect_type] = collected_types.get(defect_type, 0) + 1
        save_targets.append((case_dir, f"Type_{defect_type}"))

    # --- (B) 실패(오진/미탐) 및 엇갈림 사례 수집 ---
    if not fails_done:
        if gt_label == 0 and (sobel_pred == 1 or padim_pred == 1):
            if collected_fails['FP'] < max_fail_samples:
                collected_fails['FP'] += 1
                save_targets.append((fail_dir, f"Fail_FP(Normal_but_Defect)"))

        if gt_label == 1 and (sobel_pred == 0 or padim_pred == 0):
            if collected_fails['FN'] < max_fail_samples:
                collected_fails['FN'] += 1
                save_targets.append((fail_dir, f"Fail_FN(Defect_but_Missed)"))

        if sobel_pred != padim_pred:
            if collected_fails['DIFF'] < max_fail_samples:
                collected_fails['DIFF'] += 1
                save_targets.append((fail_dir, f"Fail_DIFF(Sobel_{sobel_pred}_vs_PaDiM_{padim_pred})"))

    # --- (C) 시각화 렌더링 및 파일 저장 ---
    if save_targets:
        padim_overlay = draw_heatmap_overlay(img_rgb, padim_score_map)
        sobel_overlay = draw_heatmap_overlay(img_rgb, sobel_res["score_map"])

        fig, axes = plt.subplots(1, 4, figsize=(20, 5))
        axes[0].imshow(img_rgb); axes[0].set_title(f"Original (GT: {gt_label})", fontsize=14, fontweight='bold', pad=10); axes[0].axis('off')
        axes[1].imshow(mask_numpy, cmap='gray'); axes[1].set_title("Ground Truth Mask", fontsize=14, fontweight='bold', pad=10); axes[1].axis('off')
        axes[2].imshow(padim_overlay); axes[2].set_title(f"PaDiM (Pred: {padim_pred})", fontsize=14, fontweight='bold', pad=10); axes[2].axis('off')
        axes[3].imshow(sobel_overlay); axes[3].set_title(f"Sobel (Pred: {sobel_pred})", fontsize=14, fontweight='bold', pad=10); axes[3].axis('off')
        plt.tight_layout()

        base_name = os.path.basename(img_path)
        for target_dir, prefix in save_targets:
            save_path = os.path.join(target_dir, f"{prefix}_{base_name}")
            plt.savefig(save_path, dpi=300, bbox_inches='tight')

        plt.close()
        print(f" -> 이미지 추출 및 저장 완료: {base_name}")

print(f"\n시각화 자료가 구글 드라이브 '/content/drive/MyDrive/CV/activeLearning/PPT_Visualizations' 경로에 저장되었습니다.")

1. 정량 비교 그래프 생성 중...
 -> 정량 비교 그래프(1_Quantitative_Graphs.png) 저장 완료.

2. 사례 비교 및 실패 사례 시각화 생성 중...
 - 각 모델별 최적 이미지 임계값 계산 중...
 - 적용된 이미지 임계값 -> Sobel: 11.4179, PaDiM: 15.7220
 -> 이미지 추출 및 저장 완료: 000.png
 -> 이미지 추출 및 저장 완료: 001.png
 -> 이미지 추출 및 저장 완료: 000.png
 -> 이미지 추출 및 저장 완료: 001.png
 -> 이미지 추출 및 저장 완료: 000.png
 -> 이미지 추출 및 저장 완료: 001.png
 -> 이미지 추출 및 저장 완료: 003.png
 -> 이미지 추출 및 저장 완료: 004.png
 -> 이미지 추출 및 저장 완료: 012.png
 -> 이미지 추출 및 저장 완료: 000.png
 -> 이미지 추출 및 저장 완료: 001.png
 -> 이미지 추출 및 저장 완료: 002.png
 -> 이미지 추출 및 저장 완료: 006.png
 -> 이미지 추출 및 저장 완료: 010.png

시각화 자료가 구글 드라이브 '/content/drive/MyDrive/CV/activeLearning/PPT_Visualizations' 경로에 저장되었습니다.
